Instalar dependência

In [ ]:
!pip install exa_py

Analisar 1 empresa (teste)

In [ ]:
from exa_py import Exa
import os
from openai import OpenAI

# 👉 COLOQUE SUAS CHAVES AQUI
os.environ["OPENAI_API_KEY"] = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
os.environ["EXA_API_KEY"] = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

# Inicializa clientes
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
exa = Exa(api_key=os.getenv("EXA_API_KEY"))

# Etapa 1: Buscar players do mercado
search_term = "mercado de nutrição bovina no Brasil"
search_response = exa.search_and_contents(
    search_term,
    highlights={"num_sentences": 2},
    num_results=10
)

companies = search_response.results

for c in companies:
    print(c.title + ':' + c.url)

# Etapa 2: Analisar a primeira empresa
c = companies[0]

all_contents = ""
search_response = exa.search_and_contents(c.url, type="keyword", num_results=5)

for r in search_response.results:
    all_contents += r.text

# Prompt
SYSTEM_MESSAGE = """
Você é um assistente sênior escrevendo um relatório de pesquisa sobre o mercado de nutrição bovina no Brasil.

1. Introdução e resumo da empresa
2. Prós e contras
3. Análise do mercado e sugestões
"""

# Chamada nova da OpenAI
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": all_contents},
    ]
)

summary = response.choices[0].message.content

print(f"\nResumo para {c.url}:\n")
print(summary)

Suplementação impulsiona produtividade e qualidade da carne bovina no Brasil - Portal:https://boiapasto.com.br/suplementacao-impulsiona-produtividade-e-qualidade-da-carne-bovina-no-brasil/
Nutrição - Portal Embrapa:https://www.embrapa.br/qualidade-da-carne/carne-bovina/producao-de-carne-bovina/nutricao
Entenda o Mercado de Carne Bovina no Brasil:https://nagro.com.br/blog/mercado-de-carne-bovina-2/
Brasil abre novo mercado para nutrição animal - Agrolink:https://www.agrolink.com.br/noticias/brasil-abre-novo-mercado-para-nutricao-animal_503949.html
Mercado da carne bovina: o que esperar e como se preparar:https://rehagro.com.br/blog/mercado-da-carne-bovina-perspectivas-producao/
Suplementação de bovinos de corte garante mais carcaça e acabamento:https://www.comprerural.com/suplementacao-de-bovinos-de-corte-garante-mais-carcaca-e-acabamento/
Recordes de produção e exportações marcaram o mercado de carne bovina em 2025:https://feedfood.com.br/recordes-de-producao-e-exportacoes-marcaram-o-m

In [ ]:
!pip install gradio exa_py openai -q

import gradio as gr
from exa_py import Exa
import openai
from google.colab import userdata

openai.api_key = userdata.get('OPENAI_API_KEY')
EXA_API_KEY = userdata.get('EXA_API_KEY')
exa = Exa(api_key=EXA_API_KEY)

SYSTEM_MESSAGE = """
Você é um assistente sênior escrevendo um relatório de pesquisa de mercado no Brasil.
1. Primeiro parágrafo: introdução e resumo da empresa.
2. Segundo parágrafo: prós e contras.
3. Terceiro parágrafo: análise do mercado e recomendações.
"""

def analisar_mercado(search_term, num_results):
    output = ""
    search_response = exa.search_and_contents(
        search_term,
        highlights={"num_sentences": 2},
        num_results=int(num_results)
      )
    companies = search_response.results

    output += "**Empresas encontradas:**\n"
    for c in companies:
          output += f"- {c.title}: {c.url}\n"
    output += "\n---\n\n"

    for c in companies:
          all_contents = ""
          res = exa.search_and_contents(c.url, type="keyword", num_results=5)
          for r in res.results:
              all_contents += r.text

          messages = [
              {"role": "system", "content": SYSTEM_MESSAGE},
              {"role": "user", "content": all_contents},
          ]
          response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
          summary = response.choices[0].message.content
          paragraphs = [p.strip() for p in summary.split('\n\n') if p.strip()]

          introduction    = paragraphs[0] if len(paragraphs) > 0 else "Informação não disponível"
          pros_cons       = paragraphs[1] if len(paragraphs) > 1 else "Informação não disponível"
          market_analysis = paragraphs[2] if len(paragraphs) > 2 else "Informação não disponível"
          recommendations = paragraphs[3] if len(paragraphs) > 3 else "Informação não disponível"

          output += f"## {c.title}\n"
          output += f"**URL:** {c.url}\n\n"
          output += f"### Introdução:\n{introduction}\n\n"
          output += f"### Prós e Contras:\n{pros_cons}\n\n"
          output += f"### Análise do Mercado:\n{market_analysis}\n\n"
          output += f"### Recomendações:\n{recommendations}\n\n"
          output += "---\n\n"

    return output

interface = gr.Interface(
      fn=analisar_mercado,
      inputs=[
          gr.Textbox(label="Termo de Busca", value="mercado de nutrição bovina no Brasil"),
          gr.Slider(minimum=1, maximum=15, value=5, step=1, label="Número de Empresas"),
      ],
      outputs=gr.Markdown(label="Relatório de Mercado"),
      title="Agent AI - Análise de Mercado",
      description="Insira um setor de mercado para analisar as principais empresas do Brasil.",
  )

interface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6099b592b4120ea5f9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
